In [0]:
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("catalog", "real-time-streaming-lakehouse")
dbutils.widgets.text("schema", "ecommerce-events")
dbutils.widgets.text("volume_path", "/Volumes/real-time-streaming-lakehouse/ecommerce-events/ecommerce-events")

In [0]:
CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
VOLUME_PATH = dbutils.widgets.get("volume_path")

In [0]:
BRONZE_TABLE = f"`{CATALOG}`.`{SCHEMA}`.`bronze-events`"
SILVER_TABLE = f"`{CATALOG}`.`{SCHEMA}`.`silver-events`"
SILVER_CHECKPOINT = f"{VOLUME_PATH}/checkpoints/silver"


In [0]:
silver_df = (
    spark.readStream.table(BRONZE_TABLE)
)

In [0]:
silver_df = (
    silver_df.filter(F.col("user_id").isNotNull())
             .filter(F.col("event_timestamp").isNotNull())
)

In [0]:
silver_df = silver_df.withColumn(
    "event_timestamp",
    F.to_timestamp(F.col("event_timestamp"))
).withWatermark("event_timestamp","10 minutes")

In [0]:
silver_df = silver_df.dropDuplicates(["user_id", "event_type", "event_timestamp", "product_id"])
silver_df = silver_df.withColumn(
    "processed_at",
    F.current_timestamp()
)

In [0]:
query =( 
        silver_df.writeStream.format("delta")
                             .outputMode("append")
                             .option("checkpointLocation", SILVER_CHECKPOINT)
                             .option("mergeSchema", "true")
                             .trigger(availableNow=True)
                             .toTable(SILVER_TABLE)

)

In [0]:
%sql
select * from `real-time-streaming-lakehouse`.`ecommerce-events`.`silver-events`
order by event_timestamp desc

amount,event_timestamp,event_type,page,product_category,product_id,session_id,user_id,_rescued_data,ingested_at,source,processed_at
0.0,2026-06-22T15:17:20.395Z,logout,checkout,food,7875,c574c9b3-da26-4e25-96c7-6254e117d4de,f8c2fbfe-11b8-4dd0-be57-414db1e0f449,null,2026-06-22T15:18:52.341Z,/Volumes/real-time-streaming-lakehouse/ecommerce-events/raw_events/events_1782141440_8.json,2026-06-22T15:20:02.764Z
41.03,2026-06-22T15:17:11.395Z,purchase,checkout,books,9657,fd0f9f05-ab37-4e25-81a7-157bcf3f62d0,f7dc1b5d-ff19-4ec5-b52d-69442cf7ce81,null,2026-06-22T15:18:52.341Z,/Volumes/real-time-streaming-lakehouse/ecommerce-events/raw_events/events_1782141440_8.json,2026-06-22T15:20:02.764Z
0.0,2026-06-22T15:17:05.395Z,login,product,books,8712,6540e545-1e65-4bc0-98ab-5ca7ab4535c7,98ef6d27-c56a-4be8-95dc-4f5835eb7675,null,2026-06-22T15:18:52.341Z,/Volumes/real-time-streaming-lakehouse/ecommerce-events/raw_events/events_1782141440_8.json,2026-06-22T15:20:02.764Z
0.0,2026-06-22T15:16:59.006Z,logout,cart,clothing,8355,38ff2615-b775-4593-ae79-f64a8f3cc0ad,c5d982db-7a4f-4264-8d85-20907ea2b942,null,2026-06-22T15:18:52.341Z,/Volumes/real-time-streaming-lakehouse/ecommerce-events/raw_events/events_1782141425_5.json,2026-06-22T15:20:02.764Z
0.0,2026-06-22T15:16:54.108Z,logout,checkout,books,4542,f684eac0-ed1b-421d-85f0-2b6af4e86a21,0ee9a89c-615b-4f98-b99d-b4e26e6154cf,null,2026-06-22T15:18:52.341Z,/Volumes/real-time-streaming-lakehouse/ecommerce-events/raw_events/events_1782141430_6.json,2026-06-22T15:20:02.764Z
0.0,2026-06-22T15:16:48.503Z,click_nav,cart,books,7780,158491bd-7965-4433-8592-0d95388f66b1,ef38a433-163a-4071-890b-514dbcb4e032,null,2026-06-22T15:18:52.341Z,/Volumes/real-time-streaming-lakehouse/ecommerce-events/raw_events/events_1782141445_9.json,2026-06-22T15:20:02.764Z
0.0,2026-06-22T15:16:48.220Z,click_nav,checkout,clothing,8432,c6a541cf-29b3-458e-991b-31237e05d3de,190be4c3-ca77-49b0-a6d5-8499acb0523c,null,2026-06-22T15:18:52.341Z,/Volumes/real-time-streaming-lakehouse/ecommerce-events/raw_events/events_1782141435_7.json,2026-06-22T15:20:02.764Z
0.0,2026-06-22T15:16:45.612Z,logout,product,books,3333,7aaf28fb-5ae2-4e68-9826-308d7e063121,572e790b-336e-4839-980c-a0688f59ca26,null,2026-06-22T15:18:52.341Z,/Volumes/real-time-streaming-lakehouse/ecommerce-events/raw_events/events_1782141409_2.json,2026-06-22T15:20:02.764Z
0.0,2026-06-22T15:16:37.497Z,logout,product,books,4912,c7e7fb57-ae09-4096-a7eb-f23554ad2a25,bd56d385-a048-48fb-a194-6f493e3f5b45,null,2026-06-22T15:18:52.341Z,/Volumes/real-time-streaming-lakehouse/ecommerce-events/raw_events/events_1782141404_1.json,2026-06-22T15:20:02.764Z
38.13,2026-06-22T15:16:36.613Z,purchase,cart,clothing,5877,a7134072-f006-452c-9310-dd5912d9e67b,56e0f15f-e11c-4010-a66e-f6e6c566ade1,null,2026-06-22T15:18:52.341Z,/Volumes/real-time-streaming-lakehouse/ecommerce-events/raw_events/events_1782141409_2.json,2026-06-22T15:20:02.764Z
